# A04 · LLMs for Business Decisions

**IMB 2026 · MADA · Workshop A04 (Mon 22 Jun 2026, 9:00–12:00)**

Adriatica's board has discovered AI and wants it *everywhere*. Your job today: find
out what large language models are **actually** good for in management — hands-on,
not from headlines.

You will work in **two tabs**:

1. **[Gemini](https://gemini.google.com)** (free, your Google account) — where you talk to the model
2. **this notebook** — your artifacts, your verification code, your notes

By the end you will be able to:

- use an LLM for summarising, extraction, classification and drafting on business documents,
- iterate on prompts deliberately and judge output quality,
- explain tokens, context windows and hallucination in management terms,
- **verify an LLM's claims against data with pandas** instead of trusting them,
- map where LLMs belong — and don't — in business decisions (risks, human-in-the-loop, EU AI Act basics).

In [ ]:
import os
import pandas as pd

DATA_BASE_URL = ""  # INSTRUCTOR: paste the published data folder URL here before class (see instructor-notes.md).

def load_file(name, reader=pd.read_csv, **kw):
    """Load a course data file: published URL -> local file -> Colab upload."""
    if DATA_BASE_URL:
        return reader(DATA_BASE_URL.rstrip("/") + "/" + name, **kw)
    if os.path.exists(name):
        return reader(name, **kw)
    try:
        from google.colab import files
        print(f"'{name}' not found - please upload it now (from this workshop's data/ folder).")
        files.upload()
        return reader(name, **kw)
    except ImportError as exc:
        raise FileNotFoundError(name) from exc

# As in A01: the export contains duplicate rows - drop them before any analysis.
orders = load_file("orders.csv").drop_duplicates()
print("orders loaded:", orders.shape)

## 1 · What LLMs are good at

### 1a · Summarise

Below is Adriatica's annual management report. Copy it into Gemini with this prompt,
then **iterate twice**: once on *format* ("as a table of 5 rows: topic | key fact"),
once on *audience* ("rewrite for the sales team — what should they do differently?").

> **Prompt:** *You are an analyst. Summarise this report in 5 bullet points for the
> board. Each bullet ≤ 15 words.*

---

# Adriatica — Management Report FY2024/25 (July 2024 – June 2025)

## Overview

Adriatica closed financial year 2024/25 with total revenue of EUR 815,630
across 4,614 orders from 1,093 active customers. The average order
value for the year was EUR 176.77. Revenue in the second half of the year reached
EUR 420,182, compared with EUR 395,447 in the first half, confirming the positive
trajectory the board set out last summer.

## Sales highlights

The strongest month of the year was December 2024, driven by the holiday
campaign, with revenue of EUR 96,506. Our largest market by
revenue remains Slovenia, and the strongest category was Electronics,
with the single best-selling product being the Compact Mirror 042.

## Operations and outlook

Fulfilment times improved over the year and customer-service load remained stable.
For FY2025/26 management proposes: (1) doubling down on the holiday-quarter
campaign that produced the December 2024 peak, (2) a loyalty programme to
lift repeat purchasing, and (3) expansion into two new EU markets, building on
the strength of our Slovenia business. The board is asked to approve the
expansion budget at the September meeting.


---

**✍️ Your answer** — Paste your best board summary here. What did the format iteration change? What did the audience iteration change?

> *(double-click this cell and write your answer here)*

**Prompt anatomy** — what you just used, named: **role** ("you are an analyst") +
**task** ("summarise in 5 bullets") + **format** ("≤ 15 words each") + optionally an
**example**. Four knobs; iterate one at a time.

### 1b · Extract structure from mess

Eight raw customer complaint emails. Ask Gemini to turn them into a **table** —
prompt: *"Extract a table with columns product | issue | severity (low/medium/high)
from these emails."* Then ask for **the same as CSV, output only the CSV**.

```
From: ana.k@example.com
Subject: Damaged on arrival
Ordered the Compact Puzzle 048 last week. The box arrived crushed and the product does not turn on.
I expect a replacement or a full refund. Order O-12345.
---
From: m.weiss@example.org
Subject: Wrong item received
I ordered a Smart Blender 019 but received some kind of kitchen scale instead. Please advise how to
return it. Mildly annoyed but mistakes happen.
---
From: luca.b@example.com
Subject: STILL WAITING
Three weeks!!! Three weeks and my Max Desk Pad 057 has not arrived. Tracking has not moved since
day 2. This is unacceptable, cancel the order if it has not shipped.
---
From: petra.n@example.org
Subject: Battery question / possible defect
The Premium Kettle 014 I bought a month ago now only lasts about 20 minutes on a charge. Is this
normal? If not I would like to claim warranty.
---
From: t.horvat@example.com
Subject: Refund not received
I returned the Classic Mirror 038 12 days ago (you confirmed receipt) but no refund has appeared on
my card. Please check.
---
From: julia.f@example.org
Subject: Small part missing
Love the Basic Organizer 024, but the small connector piece listed in the manual was not in the box.
Could you send just that part? No rush.
---
From: d.kos@example.com
Subject: Charged twice
My card statement shows two charges for one Smart Headphones 007 order. One of them needs to be
reversed, please.
---
From: s.varga@example.org
Subject: Item smells of chemicals
The Deluxe Tent 029 has a very strong chemical smell even after airing it for a week. Is it safe
for children? Considering returning it.
```

In [ ]:
# ✏️ TODO: paste the CSV Gemini gave you into the pasted_csv string below and load it with pandas — the round-trip proof: chat output became analysable data
# Hint: replace the EXAMPLE lines; keep the header



In [ ]:
import io

pasted_csv = """product,issue,severity
EXAMPLE Travel Speaker,damaged on arrival,high
EXAMPLE Desk Pad,wrong item received,medium"""
# ^ replace the example lines with the CSV Gemini produced for you (keep the header row)

complaints = pd.read_csv(io.StringIO(pasted_csv))
complaints

*(If the load failed: Gemini wrapped the CSV in prose or backticks — tell it
"output ONLY the raw CSV". Welcome to working with LLMs.)*

### 1c · Classify

Ten product reviews. Ask Gemini to label each with **sentiment** (positive / negative
/ mixed) and **topic** (product quality / delivery / service / price). Then run a
**second prompt variant** (e.g. add "explain each label in ≤ 5 words") and count
where the two runs *disagree*.

```
1. "Honestly better than expected for the price. The Max Water Bottle 032 just works." (5/5)
2. "The Max Building Set 043 broke after two weeks. Disappointed." (1/5)
3. "Delivery was fast, packaging fine. The product itself is okay, nothing special." (3/5)
4. "I WANTED to love the Classic Hair Dryer 035, and mostly I do, but the app is a mess." (3/5)
5. "Perfect gift for my nephew, the Basic Pen Set 053 kept him busy all weekend." (5/5)
6. "Not sure yet. Battery life seems short but maybe I am using it wrong?" (3/5)
7. "Customer service was rude when I asked about the Max Desk Pad 057. Product fine, service awful." (2/5)
8. "Five stars for the Basic Kettle 020, minus one for the late delivery. So four." (4/5)
9. "It does what it says. I guess that is all you can ask." (4/5)
10. "Cheap feel, flimsy buttons, returned it the next day. Avoid the Premium Puzzle 044." (1/5)
```

**✍️ Your answer** — How many labels disagreed between your two runs? Which reviews were genuinely ambiguous — and would two humans have agreed on them?

> *(double-click this cell and write your answer here)*

## 2 · Where it breaks

**Open a fresh Gemini chat** (no report pasted — that matters). Ask it these three
questions about Adriatica, *as if it knew*:

1. *Which month was Adriatica's best by revenue in FY2024/25, and what was the revenue?*
2. *What was Adriatica's single best-selling product by revenue in FY2024/25?*
3. *What was Adriatica's average order value in FY2024/25?*

Adriatica is our fictional company — **Gemini has never seen its data.** Note what it
does: refuse? hedge? or answer with confident, specific, plausible numbers?

**✍️ Your answer** — Record the three answers (or refusals) here, word for word if you can.

> *(double-click this cell and write your answer here)*

**Why would it answer at all?** An LLM is not a database. It generates the *most
plausible next words* given your prompt — from patterns in its training data, within
a limited **context window** (what's in the conversation), one **token** (word piece)
at a time. Plausible ≠ true. When the data isn't in the context, a confident answer
is a **hallucination** — and in 1a the data *was* in the context, which is why
summaries mostly work while "facts from nowhere" don't.

## 3 · The analyst who can verify

You have `orders.csv` and you have pandas (A01). The LLM made claims; **check them.**

In [ ]:
# ✏️ TODO: compute the true answers: best month by revenue, top product by revenue, average order value
# Hint: A01 patterns: groupby month / product_name on a revenue column; .idxmax(); .mean()



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
orders["revenue"] = orders["quantity"] * orders["unit_price"]
orders["month"] = pd.to_datetime(orders["order_date"]).dt.to_period("M")

best_month = orders.groupby("month")["revenue"].sum().idxmax()
top_product = orders.groupby("product_name")["revenue"].sum().idxmax()
avg_order_value = orders["revenue"].mean()

print("best month:      ", best_month)
print("top product:     ", top_product)
print("avg order value: ", round(avg_order_value, 2), "EUR")
```

</details>

**✍️ Your answer** — Score the LLM: how many of its three claims were right? And the report from 1a contains the same three facts — were THOSE right? (Yes — but now you've verified rather than assumed.)

> *(double-click this cell and write your answer here)*

**The signature lesson of this course:**

> *The analyst who can verify beats the analyst who can only ask.*

Friday you learned to distrust amazing R² scores (**trust but verify**). Same refrain
today: an LLM's fluency is not evidence. You — with twenty lines of pandas — are the
fact-checker the board doesn't know it needs.

## 4 · From chatting to building

Chat is fine for one document. For **800 complaint emails a month** you script it.
Colab has free built-in Gemini access from Python — `google.colab.ai`, no API key.
*(Pre-written — run both cells. Free tier has monthly limits, so we spend calls
sparingly: pairs can share one run.)*

In [ ]:
complaint_emails = """From: ana.k@example.com
Subject: Damaged on arrival
Ordered the Compact Puzzle 048 last week. The box arrived crushed and the product does not turn on.
I expect a replacement or a full refund. Order O-12345.
---
From: m.weiss@example.org
Subject: Wrong item received
I ordered a Smart Blender 019 but received some kind of kitchen scale instead. Please advise how to
return it. Mildly annoyed but mistakes happen.
---
From: luca.b@example.com
Subject: STILL WAITING
Three weeks!!! Three weeks and my Max Desk Pad 057 has not arrived. Tracking has not moved since
day 2. This is unacceptable, cancel the order if it has not shipped.
---
From: petra.n@example.org
Subject: Battery question / possible defect
The Premium Kettle 014 I bought a month ago now only lasts about 20 minutes on a charge. Is this
normal? If not I would like to claim warranty.
---
From: t.horvat@example.com
Subject: Refund not received
I returned the Classic Mirror 038 12 days ago (you confirmed receipt) but no refund has appeared on
my card. Please check.
---
From: julia.f@example.org
Subject: Small part missing
Love the Basic Organizer 024, but the small connector piece listed in the manual was not in the box.
Could you send just that part? No rush.
---
From: d.kos@example.com
Subject: Charged twice
My card statement shows two charges for one Smart Headphones 007 order. One of them needs to be
reversed, please.
---
From: s.varga@example.org
Subject: Item smells of chemicals
The Deluxe Tent 029 has a very strong chemical smell even after airing it for a week. Is it safe
for children? Considering returning it.
"""

In [ ]:
# Programmatic prompting with Colab's built-in (free) Gemini access - no API key.
# NOTE: free tier has monthly limits; we use only a few calls today.
try:
    from google.colab import ai
    prompt = """Extract a CSV table with columns product,issue,severity from these
customer emails. Output ONLY the CSV, no explanation.

""" + complaint_emails
    result = ai.generate_text(prompt)
    print(result)
except ImportError:
    print("(google.colab.ai is only available inside Google Colab - skipping here)")

In [ ]:
try:
    from google.colab import ai  # noqa: F401
    complaints_auto = pd.read_csv(io.StringIO(result))
    print(complaints_auto)
except ImportError:
    print("(skipped outside Colab)")
except Exception as e:
    print("Parsing failed - LLM output was not clean CSV. This is normal; tighten the prompt.")
    print(type(e).__name__, str(e)[:100])

Same prompt anatomy, now in a loop-able cell: volume, repeatability, integration
into reports. *That* is when LLM use stops being a toy — and when verification
(Section 3) stops being optional.

## 5 · Where LLMs belong in decisions

### 5a · Use-case map

**✍️ Your answer** — For each function — marketing, HR, finance, operations — write ONE high-value LLM use and ONE inappropriate use at Adriatica. (Eight lines total.)

> *(double-click this cell and write your answer here)*

### 5b · The risk list, from today's evidence

- **Hallucination** — you watched it (Section 2). Mitigation: ground it (give it the data) + verify.
- **Confidentiality** — what you paste into a free chat tool leaves the company. Personal
  data, unpublished financials, classmates' graded work: **no**. (This is why the homework
  asks you to *declare* AI use, not hide it.)
- **Bias** — the model inherits its training data's patterns; HR screening is where this bites hardest.
- **Over-reliance** — fluent ≠ correct; the danger grows as the output gets better.

### 5c · Human-in-the-loop + the law

Pattern for any consequential use: **AI drafts → human verifies (against data!) →
human decides → decision is logged with its evidence.**

EU context, in two lines: the **EU AI Act** regulates AI by *risk level* — most of
what you did today is minimal/limited risk (transparency duties: people should know
AI was involved), while HR screening or credit scoring sit in *high-risk* territory
with real obligations. The habit that keeps you on the right side of all of it is the
one you practised today: human verification, documented.

**✍️ Your answer** — Apply it: Gemini drafts Adriatica's new customer-refund policy. List 4 things a human MUST verify before it ships, and who should sign off.

> *(double-click this cell and write your answer here)*

## 6 · Wrap-up

Today: LLMs summarise, extract, classify and draft impressively **when the data is in
front of them**; they hallucinate confidently when it isn't; and you can tell the
difference, because you can verify.

**Next week (A05, Wed 1 Jul):** the AI writes **code** for your MADA project. Same
rule applies — you'll verify its code against numbers you already trust. Bring your
project group and your project data.

## 7 · Going further (optional) — search by *meaning*, not keywords

*For fast finishers — or at home. Everything below is free and runs inside this
notebook; nothing later depends on it.*

A support-desk moment: a customer writes *"my parcel arrived shattered."* Has this
happened before? Ctrl+F through last month's reviews for "shattered" finds **nothing**
— people wrote "in pieces", "cracked", "fell apart". Keyword search reads letters;
you need something that reads *meaning*.

That something is an **embedding**: a model turns each text into a list of numbers so
that texts with similar meaning get nearby numbers. A **vector database** stores those
vectors and finds the nearest ones to your question. This is the retrieval half of
every "chat with your company documents" product (the industry calls it **RAG**) — and
it is the missing link between today's two lessons: in Section 2 the LLM hallucinated
*because it had no data*; retrieval is how real systems put the right data in front
of it.

We'll use **Qdrant** — a real vector database with a free local mode: it runs entirely
inside this notebook. No server, no account, no API key.

In [ ]:
%pip install -q "qdrant-client[fastembed]"
# ^ one-time per Colab session (~1 minute). Free, no account, no API key.

In [ ]:
# Last month's reviews (synthetic, like everything at Adriatica). Pre-written - just run it.
last_month_reviews = [
    'The Premium Puzzle 044 arrived in pieces, the box was completely crushed.',
    'Glass front of the Classic Hair Dryer 035 was cracked when I unpacked it.',
    'Box looked fine but the handle of the kettle inside had snapped off.',
    'My Max Building Set 043 fell apart after a single evening of play.',
    'The seams came apart the first time I used it.',
    'One corner of the parcel was smashed and so was the mirror inside.',
    'Three weeks for delivery is simply not acceptable.',
    'Courier kept postponing, the parcel showed up ten days late.',
    'Ordered it for a birthday and it arrived long after the party.',
    'Shipping took forever even though the site promised 48 hours.',
    'Still waiting for the second half of my order.',
    'Tracking said delivered while the parcel was nowhere to be found.',
    'Arrived a day early and neatly packed.',
    'Fast shipping, no complaints there.',
    'Smooth delivery and a lovely unboxing experience.',
    'Support never answered either of my two emails.',
    'The refund took two months and three phone calls.',
    'Chat agent closed my ticket without solving anything.',
    'I was on hold for forty minutes and then got cut off.',
    'They promised a callback that never came.',
    'Customer service replaced my faulty unit within days, brilliant.',
    'The support agent was patient and genuinely helpful.',
    'Return process was completely painless.',
    'Battery of the Travel Speaker 012 dies after barely an hour.',
    'The charge does not even last my short commute.',
    'Buttons feel flimsy, like they will give up within a month.',
    'Cheap materials for a supposedly premium product.',
    'It stopped charging in the second week.',
    'The hinge wobbles - build quality is disappointing.',
    'Great value for money, I would buy it again.',
    'Overpriced for what it actually does.',
    'Half the price of the competitors and works just as well.',
    'Caught it on discount and I am very happy.',
    'Not worth the premium label at all.',
    'The Basic Kettle 020 is quiet and quick, love it.',
    'My nephew adores the Basic Pen Set 053.',
    'Exactly as described, works perfectly.',
    'Five stars, the best purchase I made this year.',
    'The design is gorgeous and it simply works.',
    'Does its job day after day without any fuss.',
    'The manual was confusing but the product itself is fine.',
    'App pairing failed twice before it finally worked.',
    'Wrong colour delivered, decided to keep it anyway.',
    'Sizing runs small, order one size up.',
    "I received someone else's order and had to send it back.",
]
print(len(last_month_reviews), "reviews")

In [ ]:
# Build the vector index (pre-written). The first run downloads a small (~67 MB)
# embedding model and embeds the reviews - allow a minute or two on Colab.
try:
    import fastembed  # noqa: F401  (the embedding engine - comes with the install cell)
    from qdrant_client import QdrantClient, models

    MODEL = "BAAI/bge-small-en-v1.5"  # turns text into 384 numbers - "meaning coordinates"
    client = QdrantClient(":memory:")  # a real vector database, running inside this notebook

    client.create_collection(
        collection_name="reviews",
        vectors_config=models.VectorParams(
            size=client.get_embedding_size(MODEL), distance=models.Distance.COSINE),
    )
    client.upload_collection(
        collection_name="reviews",
        vectors=[models.Document(text=r, model=MODEL) for r in last_month_reviews],
        payload=[{"review": r} for r in last_month_reviews],
        ids=range(len(last_month_reviews)),
    )
    print("indexed", len(last_month_reviews), "reviews")
except ImportError:
    print("(qdrant-client/fastembed not installed - run the install cell above first. This section is optional.)")

In [ ]:
# Pre-written demo - just run it.
try:
    def search(question, top=5):
        """Ask the review pile a question in plain English."""
        hits = client.query_points(
            collection_name="reviews",
            query=models.Document(text=question, model=MODEL),
            limit=top,
        ).points
        for h in hits:
            print(f"{h.score:.3f}  {h.payload['review']}")

    # The Ctrl+F way first - keyword search for the customer's own word:
    keyword_hits = [r for r in last_month_reviews if "shattered" in r.lower()]
    print("keyword hits for 'shattered':", keyword_hits)
    print()

    # The meaning way - the customer's complaint, verbatim:
    search("my parcel arrived shattered")
except (NameError, ImportError):
    print("(run the install + index cells above first. This section is optional.)")

Zero keyword hits — yet the meaning search found the crushed box, the smashed mirror
and the cracked glass, although they share **no important words** with the query: the
embedding placed "arrived shattered" and "arrived in pieces" close together because
they *mean* nearly the same thing.

Look closer, though: two of the five hits are *other* parcel problems — a wrong order
even lands near the top, and a lost parcel is in there too. The database measures
**similarity, not truth**: it read "trouble with an arriving parcel" in all of them.
The course refrain applies even here: a high score is a lead, not a verdict.

In [ ]:
# ✏️ TODO: the board asks: "are customers unhappy with our delivery times?" — query the review pile, read the top hits and judge: real pattern or one-off?
# Hint: search("...") — describe the complaint in plain English, e.g. delivery took too long



<details><summary><b>💡 Show solution</b> (try it yourself first — ask Gemini Learn Mode for a hint before opening this)</summary>

```python
search("complaints about slow delivery and parcels arriving late", top=10)

# The top hits are all delivery complaints, from several DIFFERENT customers
# -> a pattern, not a one-off.
# Watch the scores fall: below roughly 0.70 the results drift off-topic (a wrong
# order, even a POSITIVE delivery review) - that drop is your "how many hits are
# really about my question?" signal. And one genuinely late delivery ("ordered it
# for a birthday...") hides far down the list: meaning search is a better net than
# Ctrl+F, not a perfect one. Before the board acts on this, count the orders
# actually affected (pandas, Section 3 style).
```

</details>

**✍️ Your answer** — Where would meaning-search earn its keep at Adriatica — support-ticket triage? finding similar past complaints? FAQ lookup? Name one place, and one catch (hint: *similar* is not *true* — the refrain applies here too).

> *(double-click this cell and write your answer here)*

*If your MADA project data includes free text (reviews, emails, tickets), this
pattern is a legitimate stretch goal for the project — bring it up in A05.*